In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
from IPython.display import display
import ipywidgets as widgets

In [5]:
# Collapsible output panel for import/setup logs
import_messages_out = widgets.Output(
    layout=widgets.Layout(max_height="220px", overflow_y="auto")
)
import_messages_widget = widgets.Accordion(children=[import_messages_out])
import_messages_widget.set_title(0, "Import and Setup Messages")
import_messages_widget.selected_index = None

def log_import_message(message: str) -> None:
    """Route setup/import logs to the collapsible message panel."""
    with import_messages_out:
        print(message)



In [6]:
# display(import_messages_widget)


In [7]:
SKIP_CHECKS = True


In [8]:

if SKIP_CHECKS:
    log_import_message("Skipping environment checks and package installation (SKIP_CHECKS=True).")
    IS_VOILA = True
else:
    import os
    import subprocess
    import sys

    INSTALL_PACKAGE = False  # Set to True to always install, False to skip installation
    # INSTALL_PACKAGE = os.getenv("INSTALL_PACKAGE", "0").strip().lower() in {
    #     "1",
    #     "true",
    #     "yes",
    # }

    # Check if we're in Voila (where napari might not work anyway)
    """Checks if the current notebook is running in Voilà."""
    IS_VOILA =  os.environ.get('SERVER_SOFTWARE', '').startswith('voila') or 'voila' in sys.modules or os.getenv('JUPYTER_PLATFORM_DIRS') is not None

    if IS_VOILA:
        print("Running in Voilà!")
    else:
        print("Running in Jupyter (or another environment)")

    # Use collapsible widget logger when available
    log_fn = globals().get("log_import_message", print)

    # Always install in regular Jupyter (napari needs the package)
    # Skip in Voila (napari won't work there anyway)
    if not IS_VOILA and INSTALL_PACKAGE:
        log_fn("Installing package in editable mode for napari support...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '..'], check=False)
        log_fn("Installation complete.")
    else:
        log_fn("Running in Voila - package installation skipped (napari unavailable in this context)")


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
from skimage import measure, filters
import tifffile as tiff
import sys
import traceback
import os


def add_module_to_path() -> None:
    # Determine the threshold_activity_classifier directory.
    current = Path(os.getcwd())
    src = None

    while current != current.parent:
        if (current / 'utils').exists() and (current / 'core').exists() and (current / 'notebooks').exists():
            src = current
            break
        current = current.parent

    if src is None:
        src = Path(os.getcwd()).parent
        if not ((src / 'utils').exists() and (src / 'core').exists()):
            src = Path(os.getcwd()).parent.parent

    log_fn = globals().get("log_import_message", print)
    log_fn(f"SRC path: {src}")
    log_fn(f"CWD: {Path.cwd()}")
    log_fn(f"utils exists: {(src / 'utils').exists()}")
    log_fn(f"core exists: {(src / 'core').exists()}")

    sys.path.insert(0, str(src))
    sys.path.insert(0, str(Path(src).parent))

    log_fn("sys.path updated successfully")
    log_fn(f"Python path entries: {sys.path[:3]}")


try:
    import threshold_activity_classifier
except ImportError:
    add_module_to_path()




In [9]:
import matplotlib.pyplot as plt

sns.set(style="whitegrid")

from glob import glob


In [ ]:
from threshold_activity_classifier.utils.io import find_label_from_mcherry_path, find_brightfield_from_mcherry_path, find_activity_from_mcherry_path, get_images_from_mcherry
from threshold_activity_classifier.core.classifier import create_activity_labeled_image
from threshold_activity_classifier.ui import NapariImageViewer
from threshold_activity_classifier.utils.df_utils import add_meta_info


In [11]:
# Event handlers for directory selection and data preparation

import shutil
import re

try:
    from src.utils.file_utils import list_all_files, rename_all_files, copy_without_split_dict, ConfigurableFileHandler
except ImportError:
    from threshold_activity_classifier.utils.file_utils import list_all_files, rename_all_files, copy_without_split_dict, ConfigurableFileHandler



Running in Jupyter (or another environment)


In [13]:
workflow_accordion = widgets.Accordion(children=[
    widgets.HTML(value="""
    <div style='max-width:900px; padding:10px; font-family: sans-serif;'>
    <p>This notebook classifies cell instance segmentation labels as <b>active</b> or <b>dead</b> cells 
    based on intensity values from wavelength 2 images.</p>
    <h3>Workflow:</h3>
    <ol>
        <li>Choose directory containing control and sample images</li>
        <li>Set parameters in the configuration cell</li>
        <li>Run all cells to process images and generate results</li>
        <li>Review classification plots and statistics</li>
        <li>Exported results include per-instance data, image summaries, and overall statistics</li>
    </ol>
    </div>
    """)
])
workflow_accordion.set_title(0, 'Cell Activity Classification - Overview & Workflow')
workflow_accordion.selected_index = None  # collapsed by default



In [14]:
## Input directory selection widgets
if not IS_VOILA:
    display(workflow_accordion)

Accordion(children=(HTML(value="\n    <div style='max-width:900px; padding:10px; font-family: sans-serif;'>\n …

In [ ]:
# Parameters - edit as needed
data_root = Path("../../../data")

# Default directories for each data type
RAW_DIR = data_root / "Plate 2426"
PREPROCESSED_DIR = data_root / "Plate 2426_new_preprocessed_2D_split"
SPLIT_DIR = data_root / "Plate 2426_new_preprocessed_2D_split/subset"

# Current working directories (will be set by widgets)
input_dir = SPLIT_DIR
output_dir = data_root / "Plate 2426_new_preprocessed_2D_split/activity"

# File patterns
brightfield_pattern = "*_BF.tif"   # pattern for brightfield images (default : wavelength 1)
mcherry_pattern = "*_mCherry.tif"   # pattern for mCherry images (default : wavelength 2)
label_suffix = "_Cells.tif"  # expected suffix for instance labels

# Negative control sample IDs (for splitting)
NEGATIVE_CONTROL_IDS = ["A01", "A02", "H11", "H12"]  # adjust based on plate layout



# Z-slice filtering (z0 is always excluded; slices above MAX_Z_SLICES are ignored)
MAX_Z_SLICES = 20


In [16]:
def display_widgets(widget_list):
    """Helper function to display a list of widgets in a vertical layout."""
    for widget in widget_list:
        widget.layout.visibility = "visible"

def hide_widgets(widget_list):
    """Helper function to hide a list of widgets."""
    for widget in widget_list:
        widget.layout.visibility = "hidden"

def enable_widget(w):
    w.layout.display = ""      # restore default widget display (best)
    # or: w.layout.display = "flex"

def disable_widget(w):
    w.layout.display = "none"

In [17]:
# Discovery functions for raw and preprocessed files
import re

def discover_raw_metadata(raw_dir: Path) -> tuple:
    """Discover available time points, samples, and plates from raw directory.
    
    Returns:
        Tuple of (time_points, sample_ids, plate_numbers) as sorted lists
    """
    raw_path = Path(raw_dir).expanduser()
    if not raw_path.exists():
        return [], [], []
    
    time_points = set()
    sample_ids = set()
    plate_numbers = set()
    
    # Find all TIFF files
    all_files = list(raw_path.rglob("*.tif")) + list(raw_path.rglob("*.tiff"))
    
    for file_path in all_files:
        # Extract plate number from directory path
        plate_match = re.search(r'Plate\s*(\d+)', str(file_path))
        if plate_match:
            plate_numbers.add(f"Plate {plate_match.group(1)}")
        
        # Extract sample ID (e.g., A01, B02)
        sample_match = re.search(r'([A-Z]\d+)', file_path.name)
        if sample_match:
            sample_ids.add(sample_match.group(1))
        
        # Extract time point
        time_match = re.search(r't(\d+)_', file_path.name)
        if time_match:
            time_points.add(f"t{time_match.group(1)}")
    
    return sorted(time_points), sorted(sample_ids), sorted(plate_numbers)


def discover_preprocessed_samples(prep_dir: Path) -> tuple:
    """Discover available samples and plates from preprocessed directory.
    
    Returns:
        Tuple of (sample_ids, plate_numbers) as sorted lists
    """
    prep_path = Path(prep_dir).expanduser()
    if not prep_path.exists():
        return [], []
    
    sample_ids = set()
    plate_numbers = set()
    
    # Find all TIFF files
    all_files = list(prep_path.rglob("*.tif")) + list(prep_path.rglob("*.tiff"))

    file_handler = globals().get('file_handler', None)
    if file_handler is None:
        raise ValueError("ConfigurableFileHandler not found in global scope. Ensure it is imported and initialized before calling this function.")

    for file_path in all_files:
        # Extract plate number (e.g., p2126 or Plate 2126)
        plate_match = file_handler.extract_plate_number(file_path.name)
        if plate_match:
            plate_numbers.add(plate_match)
        
        # Extract sample ID
        sample_match = file_handler.extract_sample_id(file_path.name)
        print(f"File: {file_path.name}, Extracted sample ID: {sample_match}, Extracted plate number: {plate_match}")
        if sample_match:
            sample_ids.add(sample_match)
    
    return sorted(sample_ids), sorted(plate_numbers)


In [ ]:
# Data type selector and directory widgets

# Data type options
data_type_selector = widgets.RadioButtons(
    options=[
        ('Raw files (requires preprocessing)', 'raw'),
        ('Preprocessed files (requires splitting)', 'preprocessed'),
        ('Split files (ready for analysis)', 'split')
    ],
    value='split',
    layout=widgets.Layout(width='400px'),
    style={'description_width': '100px'}
)

# ===== RAW FILES WIDGETS =====
raw_input_dir = widgets.Text(
    value=str(RAW_DIR),
    description='Raw dir:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='800px', display='none')
)

discover_raw_btn = widgets.Button(
    description='Discover raw files',
    button_style='info',
    icon='search',
    layout=widgets.Layout(width='200px', display='none')
)
discover_raw_out = widgets.Output()

# ===== WAVELENGTH CONFIGURATION WIDGETS (NEW) =====
wavelength_w1 = widgets.Text(
    value='BF',
    description='w1 → ',
    placeholder='e.g., BF',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='200px', display='none')
)

wavelength_w2 = widgets.Text(
    value='mCherry',
    description='w2 → ',
    placeholder='e.g., mCherry',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='200px', display='none')
)

wavelength_w3 = widgets.Text(
    value='AnnexinV',
    description='w3 → ',
    placeholder='e.g., AnnexinV',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='200px', display='none')
)

plate_number_input = widgets.Text(
    value='',
    description='Plate #:',
    placeholder='e.g., 2126 (auto-detected if empty)',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='300px', display='none')
)

# Multi-select widgets for raw files
raw_time_points_select = widgets.Text(
    value='',
    description='Time points:',
    placeholder='e.g., t0,t1,t2',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px', display='none')
)

raw_samples_select = widgets.Text(
    value='',
    description='Samples (IDs):',
    placeholder='e.g., A01,B02,C03',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px', display='none')
)

raw_preprocess_output = widgets.Text(
    value=str(PREPROCESSED_DIR),
    description='Output to:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='800px', display='none')
)

raw_preprocess_btn = widgets.Button(
    description='Preprocess selected files',
    button_style='warning',
    icon='cogs',
    layout=widgets.Layout(width='220px', display='none')
)
raw_preprocess_out = widgets.Output()

raw_file_widgets = [
    # Raw files section
    widgets.HTML(value="<h3>Raw Files (Stage 1: Preprocessing)</h3>"),
    raw_input_dir,
    discover_raw_btn,
    discover_raw_out,
    widgets.HTML(value="<b>Channel Mappings (wavelengths):</b>"),
    wavelength_w1, wavelength_w2, wavelength_w3,
    plate_number_input,
    widgets.HTML(value="<b>Select metadata to filter by:</b>"),
    raw_time_points_select,
    raw_samples_select,
    raw_preprocess_output,
    raw_preprocess_btn,
    raw_preprocess_out,
]


# ===== PREPROCESSED FILES WIDGETS =====
prep_input_dir = widgets.Text(
    value=str(PREPROCESSED_DIR),
    description='Preprocessed dir:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='800px', display='none')
)

discover_prep_btn = widgets.Button(
    description='Discover preprocessed files',
    button_style='info',
    icon='search',
    layout=widgets.Layout(width='230px', display='none')
)
discover_prep_out = widgets.Output()

# Multi-select widgets for preprocessed files

neg_controls_select = widgets.Text(
    value='',
    description='Negative controls:',
    placeholder='e.g., A01,B02,H12',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px', display='none')
)

sample_samples_select = widgets.Text(
    value='',
    description='Sample cells:',
    placeholder='e.g., A03,B04,C05',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px', display='none')
)

prep_split_output = widgets.Text(
    value=str(SPLIT_DIR),
    description='Split to:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='800px', display='none')
)

prep_split_btn = widgets.Button(
    description='Split selected files',
    button_style='warning',
    icon='code-branch',
    layout=widgets.Layout(width='200px', display='none')
)
prep_split_out = widgets.Output()

prep_split_widgets = [
    # Preprocessed files section
    widgets.HTML(value="<h3>Preprocessed Files (Stage 2: Splitting)</h3>"),
    prep_input_dir,
    discover_prep_btn,
    discover_prep_out,
    widgets.HTML(value="<b>Negative controls (comma-separated):</b>"),
    neg_controls_select,
    widgets.HTML(value="<b>Sample cells (comma-separated):</b>"),
    sample_samples_select,
    prep_split_output,
    prep_split_btn,
    prep_split_out,
]

# ===== SPLIT FILES WIDGETS =====
neg_control_dir = widgets.Text(
    value=str(SPLIT_DIR / 'negative_controls'),
    description='Neg. control dir:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='800px')
)


max_z_input = widgets.BoundedIntText(
    value=MAX_Z_SLICES,
    min=1,
    max=999,
    description='Max z-slice:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='160px'),
)
set_neg_control_btn = widgets.Button(
    description='Set negative control directory',
    button_style='info',
    layout=widgets.Layout(width='240px')
)

neg_control_out = widgets.Output()

neg_control_widgets = [
    widgets.HTML(value="<h3>Negative Controls (Stage 3: Analysis)</h3>"),
    neg_control_dir,
    widgets.HBox([
        widgets.HTML('<small>z0 always excluded &nbsp;|&nbsp;</small>'),
        max_z_input,
    ]),
    set_neg_control_btn, neg_control_out]

In [19]:

def update_ui_for_data_type(change):
    """Update visible widgets based on selected data type."""
    data_type = change['new'] if change else data_type_selector.value
    
    # Hide all stage-specific widgets
    for w in raw_file_widgets + prep_split_widgets + neg_control_widgets:
        w.layout.display = 'none'
    
    if data_type == 'raw':
        # Show raw files widgets
        for w in raw_file_widgets:
            w.layout.display = ''
    elif data_type == 'preprocessed':
        # Show preprocessed files widgets
        for w in prep_split_widgets:
            w.layout.display = ''
    else:
        # Show split files widgets (only negative control directory selection)
        for w in neg_control_widgets:
            w.layout.display = ''


data_type_selector.observe(update_ui_for_data_type, names='value')

# Assemble directory widgets
dir_widgets = [
    widgets.HTML(value="<h1>Directory Selection & Data Filtering</h1>"),
    widgets.HTML(value="<p>Select the type of data you want to process:</p>"),
    data_type_selector,
] + raw_file_widgets + prep_split_widgets + neg_control_widgets

# Hide all stage-specific widgets
for w in raw_file_widgets + prep_split_widgets + neg_control_widgets:
    w.layout.display = 'none'


In [ ]:

def get_configured_file_handler():
    from threshold_activity_classifier.utils.file_utils import ConfigurableFileHandler
    """Create ConfigurableFileHandler with user-provided wavelength mappings and plate number."""
    # Build wavelength mappings from widget inputs
    wavelength_mappings = {}
    
    w1_val = wavelength_w1.value.strip()
    if w1_val:
        wavelength_mappings[1] = w1_val
    
    w2_val = wavelength_w2.value.strip()
    if w2_val:
        wavelength_mappings[2] = w2_val
    
    w3_val = wavelength_w3.value.strip()
    if w3_val:
        wavelength_mappings[3] = w3_val
    
    # Get plate number if provided
    plate_num = plate_number_input.value.strip() if plate_number_input.value.strip() else None
    
    # Create handler with user configuration
    if wavelength_mappings:
        handler = ConfigurableFileHandler(
            wavelength_mappings=wavelength_mappings,
            plate_number=plate_num
        )
    else:
        # Use default ConfigurableFileHandler (loads from YAML)
        handler = ConfigurableFileHandler(plate_number=plate_num)
    
    return handler

def on_discover_raw(btn):
    """Discover available time points, samples, and plates from raw directory."""
    with discover_raw_out:
        discover_raw_out.clear_output()
        raw_path = Path(raw_input_dir.value).expanduser()
        
        if not raw_path.exists():
            print(f"Error: Raw directory does not exist: {raw_path}")
            return
        
        print(f"Discovering files in: {raw_path}")
        time_points, samples, plates = discover_raw_metadata(raw_path)
        
        print(f"Found {len(time_points)} time points: {time_points}")
        print(f"Found {len(samples)} samples: {samples}")
        print(f"Found {len(plates)} plates: {plates}")
                
        # print(f"\nAll items pre-selected. Deselect to exclude specific items.")


def on_preprocess_raw(btn):
    """Preprocess selected raw files by copying/renaming to standardized format."""
    with raw_preprocess_out:
        raw_preprocess_out.clear_output()
        raw_path = Path(raw_input_dir.value).expanduser()
        prep_path = Path(raw_preprocess_output.value).expanduser()
        
        if not raw_path.exists():
            print(f"Error: Raw directory does not exist: {raw_path}")
            return
        
        # Get configured handler with user wavelength mappings
        file_handler = globals().get('file_handler', None)
        assert file_handler is not None, "Error: File handler not configured. Please discover raw files first to set wavelength mappings."
        
        print("═" * 70)
        print("Raw File Preprocessing Configuration")
        print("═" * 70)
        print(f"Input directory:  {raw_path}")
        print(f"Output directory: {prep_path}")
        print(f"\nWavelength mappings:")
        for wl_idx, channel_name in sorted(file_handler.wavelength_mappings.items()):
            print(f"  w{wl_idx} → {channel_name}")
        if file_handler._default_plate_number:
            print(f"Default plate number: {file_handler._default_plate_number}")
        print("═" * 70)
        
        # Get selected filters
        selected_times = set(tp.strip() for tp in raw_time_points_select.value.split(',') if tp.strip())
        selected_samples = set(s.strip() for s in raw_samples_select.value.split(',') if s.strip())
        
        if not (selected_times or selected_samples):
            print("Error: Please select at least one time point or sample.")
            return
        
        prep_path.mkdir(parents=True, exist_ok=True)
        
        print(f"\nProcessing filters:")
        print(f"  Time points: {selected_times if selected_times else 'All'}")
        print(f"  Samples: {selected_samples if selected_samples else 'All'}")
        
        # Find and filter files
        all_files = list_all_files(str(raw_path), file_handler=file_handler)
        renamed_file_tuples = rename_all_files(all_files, file_handler)
        # all_files = list(raw_path.rglob("*.tif")) + list(raw_path.rglob("*.tiff"))
        filtered_files = {}

        for file_type in renamed_file_tuples:
            print(f"Found {len(renamed_file_tuples[file_type])} files of type '{file_type}'")
            filtered_files[file_type] = []

            for src, renamed in renamed_file_tuples[file_type]:

                # Check sample filter
                if selected_samples:
                    sample_match = re.search(r'([A-Z]\d+)', renamed)
                    if not sample_match or sample_match.group(1) not in selected_samples:
                        continue
                
                # Check time filter
                if selected_times:
                    time_match = re.search(r't(\d+)_', renamed)
                    if not time_match or f"t{time_match.group(1)}" not in selected_times:
                        continue
                
                filtered_files[file_type].append((src, renamed))
        
        if not filtered_files:
            print("No files match the selected criteria.")
            return
        
        print(f"\nProcessing {len(filtered_files)} files...")
        
        # Process files using the standard pipeline
        try:
            copied = copy_without_split_dict(filtered_files, prep_path)

        except Exception as e:
            print(f"Error during preprocessing: {str(e)}")
            traceback.print_exc()
            return
        
        print(f"\nPreprocessing complete. Processed {copied} files.")
        print(f"Preprocessed files saved to: {prep_path}")
        
        # Auto-transition to preprocessed stage
        prep_input_dir.value = str(prep_path)
        data_type_selector.value = 'preprocessed'


def on_discover_prep(btn):
    """Discover available samples and plates from preprocessed directory."""
    with discover_prep_out:
        discover_prep_out.clear_output()
        prep_path = Path(prep_input_dir.value).expanduser()
        
        if not prep_path.exists():
            print(f"Error: Preprocessed directory does not exist: {prep_path}")
            return
        
        print(f"Discovering files in: {prep_path}")
        samples, plates = discover_preprocessed_samples(prep_path)
        
        print(f"Found {len(samples)} samples: {samples}")
        print(f"Found {len(plates)} plates: {plates}")
                
        print(f"\nSelect samples to split. Check 'Negative controls' checkbox for control samples.")


def on_split_by_sample(btn):
    """Split preprocessed files into negative controls and samples."""
    file_handler = globals().get('file_handler', None)
    assert file_handler is not None, "Error: File handler not configured. Please preprocess raw files first to set wavelength mappings."

    with prep_split_out:
        prep_split_out.clear_output()
        prep_path = Path(prep_input_dir.value).expanduser()
        split_path = Path(prep_split_output.value).expanduser()
        
        if not prep_path.exists():
            print(f"Error: Preprocessed directory does not exist: {prep_path}")
            return
        
        # Parse comma-separated values from text inputs
        neg_ctrl_input = neg_controls_select.value.strip()
        samples_input = sample_samples_select.value.strip()
        
        # Parse into sets (case-insensitive, stripped)
        neg_controls = set(s.strip().upper() for s in neg_ctrl_input.split(',') if s.strip())
        samples = set(s.strip().upper() for s in samples_input.split(',') if s.strip())
        
        # Check if at least one is specified
        if not (neg_controls or samples):
            print("Error: Please specify at least one negative control or sample cell.")
            return
        
        print(f"Splitting files from: {prep_path}")
        print(f"Negative control samples: {neg_controls if neg_controls else 'None'}")
        print(f"Sample cells: {samples if samples else 'None'}")
        
        # Find all files and extract available sample IDs
        all_files = list(prep_path.rglob("*.tif")) + list(prep_path.rglob("*.tiff"))
        
        if not all_files:
            print("No TIFF files found in preprocessed directory.")
            return
        
        # Extract all available sample IDs from files
        available_samples = set()
        for file_path in all_files:
            sample_id = file_handler.extract_sample_id(file_path.name)
            if sample_id:
                available_samples.add(sample_id.upper())
        
        print(f"\nAvailable samples in directory: {sorted(available_samples)}")
        
        # Validate that specified samples exist
        invalid_neg_controls = neg_controls - available_samples
        invalid_samples = samples - available_samples
        
        if invalid_neg_controls:
            print(f"Warning: Negative control samples not found: {invalid_neg_controls}")
        
        if invalid_samples:
            print(f"Warning: Sample cells not found: {invalid_samples}")
        
        # Get the valid samples to process
        valid_neg_controls = neg_controls & available_samples
        valid_samples = samples & available_samples
        
        if not (valid_neg_controls or valid_samples):
            print("Error: None of the specified samples were found in the directory.")
            return
        
        print(f"\nProcessing:")
        print(f"  - Valid negative controls: {valid_neg_controls if valid_neg_controls else 'None'}")
        print(f"  - Valid sample cells: {valid_samples if valid_samples else 'None'}")
        
        # Create output directories
        controls_dir = split_path / "negative_controls"
        samples_dir = split_path / "samples"
        controls_dir.mkdir(parents=True, exist_ok=True)
        samples_dir.mkdir(parents=True, exist_ok=True)
        
        controls_count = 0
        samples_count = 0
        unmatched = 0
        
        for src_file in all_files:
            sample_id = file_handler.extract_sample_id(src_file.name)
            
            if sample_id is None:
                unmatched += 1
                continue
            
            sample_id_upper = sample_id.upper()
            
            # Determine destination based on selection
            if sample_id_upper in valid_neg_controls:
                dst_dir = controls_dir
                controls_count += 1
            elif sample_id_upper in valid_samples:
                dst_dir = samples_dir
                samples_count += 1
            else:
                continue  # Skip unselected samples
            
            # Copy file
            dst_file = dst_dir / src_file.name
            if not dst_file.exists():
                shutil.copy2(src_file, dst_file)
        
        print(f"\nSplit complete:")
        print(f"  - Negative controls: {controls_count} files -> {controls_dir}")
        print(f"  - Samples: {samples_count} files -> {samples_dir}")
        if unmatched > 0:
            print(f"  - Unmatched files (no sample ID): {unmatched}")
        
        # Auto-transition to split stage
        neg_control_dir.value = str(controls_dir)  # Set to negative controls directory
        data_type_selector.value = 'split'


def on_set_neg_control(btn):
    """Set negative control directory and enable analysis button."""
    with neg_control_out:
        neg_control_out.clear_output()
        in_path = Path(neg_control_dir.value).expanduser()
        
        if not in_path.exists():
            print(f"Error: Input directory does not exist: {in_path}")
            print(f"Error: Negative control directory does not exist: {in_path}")
            print(f"Available directories after split:")
            split_parent = in_path.parent
            if split_parent.exists():
                for item in split_parent.iterdir():
                    if item.is_dir():
                        print(f"  - {item.name}")
            return
        
        print(f"Negative control directory set: {in_path}")

        # Export to notebook globals
        globals()['input_dir'] = in_path
        
        global image_paths
        image_paths = sorted(glob(os.path.join(str(in_path), "**", mcherry_pattern), recursive=True))

        if not image_paths:
            print(f"Warning: No images found matching pattern '{mcherry_pattern}' under {in_path}")
            print("Check the input directory and image pattern.")
            return

        # Filter z0 and slices above max_z before any processing
        from threshold_activity_classifier.utils.io import filter_paths_by_z_index
        _fh = globals().get('file_handler')
        _max_z = int(max_z_input.value)
        if _fh is not None:
            _before = len(image_paths)
            image_paths = filter_paths_by_z_index(image_paths, _fh, min_z=1, max_z=_max_z)
            print(f"Z-filter (z1–z{_max_z}): kept {len(image_paths)}/{_before} images")

        # Show sample filenames
        print(f"Found {len(image_paths)} images matching '{mcherry_pattern}'")
        
        stats_sec = globals().get('stats_section')
        if stats_sec is not None:
            stats_sec.selected_index = 0


# Attach event handlers
discover_raw_btn.on_click(on_discover_raw)
raw_preprocess_btn.on_click(on_preprocess_raw)
discover_prep_btn.on_click(on_discover_prep)
prep_split_btn.on_click(on_split_by_sample)
set_neg_control_btn.on_click(on_set_neg_control)


In [21]:
image_paths = None
file_handler = get_configured_file_handler()  # Initialize file handler with user configuration (or defaults)

In [ ]:
if not IS_VOILA:
    display(widgets.VBox(dir_widgets))


: 

In [ ]:
# import threading
# Compute stats widgets

# Label directory and suffix configuration
label_dir_input = widgets.Text(
    value="",
    description="Label dir:",
    placeholder="Leave empty to use same folder as images",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="800px"),
)

label_suffix_input = widgets.Text(
    value="_Cells",
    description="Label suffix:",
    placeholder="e.g., _Cells",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="300px"),
)

compute_stats_button = widgets.Button(description="Compute stats", button_style="primary")
# compute_stats_button.layout.visibility = "hidden"

compute_stats_out = widgets.Output()

# Outputs for the accordion panels
compute_stats_summary = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", max_height="320px", overflow="auto"))
compute_stats_plot = widgets.Output()


# Accordion containing summary and plots
stats_accordion = widgets.Accordion(children=[compute_stats_summary, compute_stats_plot])
stats_accordion.set_title(0, "Per-image Summary")
stats_accordion.set_title(1, "Distribution Plots")
stats_accordion.selected_index = None  # collapsed by default

# Ensure the accordion is shown with the compute button
compute_stats_widgets = [
    widgets.HTML("<h1>Cell Activity Statistics</h1>"),
    widgets.HTML("<b>Label configuration:</b>"),
    label_dir_input,
    label_suffix_input,
    compute_stats_button, compute_stats_out, stats_accordion]

# # Trigger population shortly after compute finishes (non-blocking)
# def _delayed_populate(btn):
#     threading.Timer(0.2, populate_stats).start()

# compute_stats_button.on_click(_delayed_populate)

# # Also update accordion after applying classification, if apply_btn is available
# if "apply_btn" in globals():
#     apply_btn.on_click(lambda b: threading.Timer(0.2, populate_stats).start())

# # If metrics_df already exists, populate immediately
# if globals().get("metrics_df", None) is not None and not globals()["metrics_df"].empty:
#     populate_stats()



In [79]:
metrics_df = None

In [ ]:
from pathlib import Path
# from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time, sys

from core.metrics import InstanceMetricsExtractor


def compute_stats(image_paths, label_paths, output=sys.stdout):

    PERCENTILES = [95, 90, 75, 50]  # keep your order

    print("Starting metrics extraction...")

    metrics_extractor = InstanceMetricsExtractor(percentiles=PERCENTILES)

    with output:
        metrics_df = metrics_extractor.process_batch_images(image_paths, lbl_paths=label_paths)

        if metrics_df.empty:
            print("No metrics were extracted. Check if the image paths are correct and the extractor is configured properly.")
            return pd.DataFrame()  # Return empty DataFrame if no metrics were extracted
        
        metrics_df = add_meta_info(metrics_df, globals().get('file_handler', None))

    return metrics_df


In [81]:
def plot_single_metric(metrics_df, metric, ax):
    data = metrics_df[metric].dropna()
    if data.empty:
        ax.text(0.5, 0.5, f'{metric} - no data', ha='center', va='center')
        ax.set_axis_off()
        return

    # sum_intensity: plot log10 values (to handle large scale)
    if metric == 'sum_intensity':
        data_pos = data[data > 0]
        if data_pos.empty:
            ax.text(0.5, 0.5, 'sum_intensity - no positive data', ha='center', va='center')
            ax.set_axis_off()
            return
        log_data = np.log10(data_pos)
        clip_val = np.percentile(log_data, 99)
        plot_data = log_data[log_data <= clip_val]
        sns.histplot(plot_data, bins=50, kde=True, stat='density', color=f'C{ax.get_subplotspec().colspan.start}', ax=ax)
        ax.set_title(f'{metric} (log10)')
        ax.set_xlabel('log10(Sum Intensity)')
        ax.set_ylabel('Density')
        median_log = np.median(log_data)
        ax.axvline(median_log, color='k', linestyle='--', linewidth=1.5)
        orig_median = 10 ** median_log
        ax.text(median_log, ax.get_ylim()[1] * 0.9, f'Median=10^{median_log:.2f}\n({orig_median:,.0f})', 
                ha='right', va='top', fontsize=9, bbox=dict(facecolor='white', alpha=0.6))
        ax.grid(alpha=0.3)

    else:
        # Other metrics: clip at 99th percentile and use log x-scale
        data_pos = data[data > 0]
        if data_pos.empty:
            print("zero values found")
        
        # Clip data at 99th percentile
        data = np.clip(data, 0, data_pos.quantile(0.99))
        sns.histplot(data, bins=50, kde=True, stat='density', color=f'C{ax.get_subplotspec().colspan.start}', ax=ax)
        # ax.set_xscale('log')
        ax.set_title(f'{metric} (clipped to 99th pct)')
        ax.set_xlabel('Intensity Value')
        ax.set_ylabel('Density')
        median_val = data_pos.median()
        ax.axvline(median_val, color='k', linestyle='--', linewidth=1.5)
        # ax.text(median_val, ax.get_ylim()[1] * 0.9, f'Median={median_val:.1f}', ha='right', va='top', fontsize=9, bbox=dict(facecolor='white', alpha=0.6))
        ax.grid(alpha=0.3)

def plot_stats(metrics_df):

    # Distribution-only metrics plot with log-scaling, median lines, and 99th-percentile clipping
    metrics_to_plot = ['mean_intensity', 'max_intensity', 'percentile_95', 'percentile_90', 'percentile_75', 'sum_intensity']
    # Keep only available metrics
    metrics_to_plot = [m for m in metrics_to_plot if m in metrics_df.columns]
    n_plots = len(metrics_to_plot)
    ncols = 3
    nrows = (n_plots + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    # Ensure axes is iterable
    if hasattr(axes, 'flatten'):
        axes_arr = axes.flatten()
    else:
        axes_arr = [axes]


    for i, metric in enumerate(metrics_to_plot):
        ax = axes_arr[i]
        plot_single_metric(metrics_df, metric, ax)

    # Remove any unused axes
    for j in range(n_plots, len(axes_arr)):
        fig.delaxes(axes_arr[j])

    fig.suptitle('Distribution plots')

    plt.tight_layout()
    plt.show()

    # Print a short summary of chosen metrics
    print('=== SELECTED METRICS SUMMARY ===')
    for metric in metrics_to_plot:
        s = metrics_df[metric].dropna()
        if s.empty:
            print(f'{metric}: no data')
            continue
        if metric == 'sum_intensity':
            s_pos = s[s > 0]
            median_val = s_pos.median() if len(s_pos) else np.nan
            print(f'{metric}: n={len(s)}, median={median_val:,.0f}, mean={s.mean():.2f}, std={s.std():.2f}')
        else:
            print(f'{metric}: n={len(s)}, mean={s.mean():.2f}, median={s.median():.2f}, std={s.std():.2f}')


In [82]:
def populate_stats(md, summary_out, plot_out):
    """Populate the summary and plot outputs from the current metrics_df."""
    with summary_out:
        summary_out.clear_output(wait=True)
        if md is None or md.empty:
            print("No metrics available. Run 'Compute stats' first.")
        else:
            try:
                display(md.head().round(2))
            except Exception as e:
                print("Error generating summary:", e)

    with plot_out:
        plot_out.clear_output(wait=True)
        if md is None or md.empty:
            return
        try:
            plot_stats(md)
        except Exception as e:
            print("Error plotting stats:", e)


In [ ]:
from functools import partial

def on_compute_stats(btn):
    global metrics_df, label_paths

    image_paths = globals()["image_paths"]

    with compute_stats_out:
        if image_paths is None:
            compute_stats_out.clear_output()
            print("Error: image_paths not set. Please set the negative control directory first and ensure images are found.")
            return

        # Resolve label directory and suffix from widgets
        raw_label_dir = label_dir_input.value.strip()
        lbl_dir = Path(raw_label_dir).expanduser() if raw_label_dir else None

        mask_suffix = label_suffix_input.value.strip() or "_Cells"

        label_paths = [
            find_label_from_mcherry_path(Path(p), mask_suffix=mask_suffix, label_dir=lbl_dir)
            for p in image_paths
        ]

        print("Computing metrics for images...")

        metrics_df = compute_stats(image_paths, label_paths, output=compute_stats_out)

        if not metrics_df.empty:

            with compute_stats_out:
                compute_stats_out.clear_output()
                print(f"Processed {len(image_paths)} images.")

                populate_stats(metrics_df, compute_stats_summary, compute_stats_plot)

            classifier_sec = globals().get('classifier_section')
            if classifier_sec is not None:
                classifier_sec.selected_index = 0
        else:
                compute_stats_out.clear_output()
                print("No metrics were computed. Check the image paths and extractor configuration.")


# compute_stats_button.on_click(partial(on_compute_stats, nxt=classifier_widgets))
compute_stats_button.on_click(on_compute_stats)



In [84]:
if not IS_VOILA:
    display(widgets.VBox(compute_stats_widgets))

In [55]:
parameter_guide_html = """
<div style='max-width:900px; padding:10px; font-family: sans-serif;'>
<ul>
  <li><b>threshold_metric</b>: Choose from "mean_intensity", "max_intensity", "sum_intensity", "percentile_90", "percentile_75", "percentile_95"</li>
  <li><b>threshold_method</b>:
    <ul>
      <li>"otsu" for automated Otsu thresholding</li>
      <li>"manual" to use the manual_threshold value</li>
      <li>numeric value (e.g., 1000) for fixed threshold</li>
      <li>"percentile" to use a specific percentile as the threshold (requires numeric input in manual_threshold)</li>
    </ul>
  </li>
  <li><b>per_image_threshold</b>: Compute OTSU threshold for each image individually (per-image) or use a global threshold across all images.</li>
</ul>
</div>
"""

param_box = widgets.Accordion(children=[
  widgets.HTML(value=parameter_guide_html)
])
param_box.set_title(0, 'Classification Parameter Guide')
param_box.selected_index = None  # collapsed by default


In [56]:
# Intensity metrics to compute and classify on
# Available options: "mean_intensity", "max_intensity", "sum_intensity", "percentile_90", "percentile_75"
threshold_metric = "percentile_90"  # metric to use for active/dead classification

# Threshold settings
per_image_threshold = False     # if True compute threshold per-image; if False compute one global threshold

# Threshold method options:
# - "otsu": automated Otsu threshold
# - "manual": use the manual_threshold value below
# - numeric value (e.g., 500): use as fixed threshold
threshold_method = "otsu"      
manual_threshold = 1000.0      # used only when threshold_method = "manual"

# Classification labels
active_label = "active"        # label for cells above threshold (bright cells)
dead_label = "dead"           # label for cells below threshold (dim cells)

In [64]:
# Interactive thresholding widget (Voila-friendly)
# Relies on existing globals: metrics_df, apply_threshold_classification, manual_threshold, threshold_metric, threshold_method, per_image_threshold

# Allowed metric choices (limit to sensible intensity metrics present in metrics_df)
_allowed_metrics = ["mean_intensity", "max_intensity", "sum_intensity", "percentile_90", "percentile_75", "percentile_95"]
# metric_options = [m for m in _allowed_metrics if m in globals().get("metrics_df", pd.DataFrame()).columns]

# Fallback to numeric columns if none of the above present
# if not metric_options:
    # metric_options = list(globals().get("metrics_df", pd.DataFrame()).select_dtypes(include=[np.number]).columns)

metric_options = _allowed_metrics

metric_dropdown = widgets.Dropdown(
    options=metric_options,
    value=globals().get("threshold_metric", metric_options[0] if metric_options else None),
    description="Metric:",
    layout=widgets.Layout(width="360px")
)

method_dropdown = widgets.Dropdown(
    options=[
        ("Otsu (per-image)", "otsu_per_image"),
        ("Otsu (global)", "otsu_global"),
        ("Manual threshold", "manual"),
        ("Percentile (numeric)", "percentile"),
    ],
    value="otsu_per_image" if globals().get("per_image_threshold", False) else "otsu_global",
    description="Method:",
    layout=widgets.Layout(width="360px")
)

manual_input = widgets.FloatText(
    value=float(globals().get("manual_threshold", 1000.0)),
    description="Manual val:",
    layout=widgets.Layout(width="360px"),
    step=1.0
)

apply_btn = widgets.Button(description="Apply threshold", button_style="primary", icon="check")
class_out = widgets.Output()

class_summary = widgets.Output()
# Accordion containing summary and plots
class_accordion = widgets.Accordion(children=[class_summary])
class_accordion.selected_index = None  # collapsed by default
class_accordion.set_title(0, "Classification Summary ")

classifier_widgets = [
    widgets.HTML("<h1>Cell Activity Thresholding</h1>"),
    param_box,
    widgets.HBox([metric_dropdown, method_dropdown]),
    manual_input,
    widgets.HBox([apply_btn]),
    class_out,
    class_accordion,

]


In [65]:
import sys
from core.classifier import ThresholdInstanceClassifier
from config.models import ThresholdConfig, ThresholdParams

In [66]:
def generate_summary(metrics_df, out_widget=sys.stdout):

    # Summary table per image with classification results
    summary = metrics_df.groupby('image').agg(
        n_instances=('label', 'count'),
        n_active=('is_active', 'sum'),
        n_dead=('is_active', lambda x: (~x).sum()),
        activity_ratio=('is_active', 'mean'),  # fraction of active cells
        metric_median=('metric_value', 'median'),
        metric_mean=('metric_value', 'mean'),
        metric_std=('metric_value', 'std'),
        threshold_used=('threshold', 'first')  # threshold used for this image
    ).reset_index()
    summary['percent_active'] = (summary['n_active'] / summary['n_instances']) * 100

    print(f"Total images processed: {len(summary)}")
    print(f"Average active cell percentage: {summary['activity_ratio'].mean()*100:.1f}% ± {summary['activity_ratio'].std()*100:.1f}%")
    with out_widget:
        print("\nPer-image summary:")
        print(summary.round(2))

    return summary

In [67]:
def _update_manual_visibility(change=None):
    method = method_dropdown.value
    manual_input.layout.display = "block" if method in ["manual", "percentile"] else "none"

# initialize visibility
_update_manual_visibility()

method_dropdown.observe(_update_manual_visibility, names="value")


In [77]:
def plot_classification_distributions(metrics_df, sel_metric):
    # Create figure and axis for plotting
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Get the metric column name from metrics_df
    metric_col = 'metric_value'
    if metric_col not in metrics_df.columns:
        # Try to find the metric column (fallback)
        metric_col = [col for col in metrics_df.columns if 'intensity' in col.lower()][0] if any('intensity' in col.lower() for col in metrics_df.columns) else 'metric_value'
    
    # Plot distributions for active vs dead cells
    active_data = metrics_df[metrics_df['is_active']][metric_col]
    dead_data = metrics_df[~metrics_df['is_active']][metric_col]
    
    # Clip extreme values at the global 99th percentile for clearer visualization
    upper = metrics_df[metric_col].dropna().quantile(0.99)
    active_plot = active_data.dropna().clip(upper=upper)
    dead_plot = dead_data.dropna().clip(upper=upper)

    if len(active_plot) > 1:
        sns.kdeplot(
            data=active_plot,
            label=active_label,
            fill=True,
            ax=ax,
            color="green",
            alpha=0.6,
        )
    if len(dead_plot) > 1:
        sns.kdeplot(
            data=dead_plot,
            label=dead_label,
            fill=True,
            ax=ax,
            color="red",
            alpha=0.6,
        )

    ax.set_xlabel(f"{sel_metric} (clipped at 99th percentile)")
    ax.set_ylabel("Density")
    ax.set_title(f"{sel_metric} distribution by classification")
    ax.legend()
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()


In [ ]:
def _on_apply(btn):
    with class_out:
        class_out.clear_output(wait=True)
        if metric_dropdown.value is None:
            print("No metric selected.")
            return
        # Map UI selections to function parameters
        sel_metric = metric_dropdown.value
        sel_method = method_dropdown.value
        if sel_method == "otsu_per_image":
            config = ThresholdConfig(
                method='otsu',
                metric=sel_metric, 
                per_image=True
            )
        elif sel_method == "otsu_global":
            config = ThresholdConfig(
                method='otsu',
                metric=sel_metric, 
            )
        else:
            try:
                man_val = float(manual_input.value)
            except Exception:
                print("Invalid percentile value. Please enter a numeric value (e.g., 90 for 90th percentile).")
                return
            if sel_method == "percentile":
                if not (0 <= man_val <= 100):
                    print("Percentile value must be between 0 and 100.")
                    return
                config = ThresholdConfig(
                    method='percentile',
                    metric=sel_metric,
                    params=ThresholdParams(percentile=man_val)
                )
            elif sel_method == "manual":
                config = ThresholdConfig(
                    method='manual',
                    metric=sel_metric,
                    params=ThresholdParams(manual_value=man_val)
                )
            else:
                print("Invalid method selected.")
                return
        
        # Export to notebook globals for downstream cells
        globals()['threshold_config'] = config

        print(f"Applying classification: metric={sel_metric}, method={sel_method}")

        classifier = ThresholdInstanceClassifier(config=config)

        try:            
            class_summary.clear_output(wait=True)

            # Re-run classification (in-place update of metrics_df)
            metrics_df = classifier.classify_instances(globals()['metrics_df'])

            # Recompute a simple per-image summary (similar to later cell)
            summary_loc = generate_summary(metrics_df, out_widget=class_summary)

            # Export summary to globals for downstream usage
            globals()['summary'] = summary_loc
            globals()['metrics_df'] = metrics_df

            # Display quick stats
            total_images = len(summary_loc)
            total_cells = len(metrics_df)
            overall_activity = metrics_df['is_active'].mean() if total_cells > 0 else 0.0

            print(f"Done. Images: {total_images}, Cells: {total_cells}, Overall activity: {overall_activity:.1%}")

            with class_summary:
                plot_classification_distributions(metrics_df, sel_metric)


            save_sec = globals().get('save_section')
            if save_sec is not None:
                save_sec.selected_index = 0

        except Exception as e:
            print("Error applying classification:", e)

apply_btn.on_click(_on_apply)



In [79]:
if not IS_VOILA:
    display(widgets.VBox(classifier_widgets))

In [96]:
from pathlib import Path

In [97]:
# Save output images button
output_dir_text = widgets.Text(
    value=str(globals().get("output_dir", "")),
    description='Output dir:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='800px')
)

save_images_btn = widgets.Button(
    description="Save activity images",
    button_style="primary",
    icon="save",
    layout=widgets.Layout(width="220px")
)

save_images_out = widgets.Output()

save_images_summary = widgets.Output()
save_images_accordion = widgets.Accordion(children=[save_images_summary])
save_images_accordion.selected_index = None  # collapsed by default
save_images_accordion.set_title(0, "Save Images Output")

save_widgets = [
    widgets.HTML("<h1>Export Activity Images</h1>"),
    output_dir_text,
    save_images_btn,
    save_images_out,
    save_images_accordion,
]

In [98]:
def save_threshold_config(config, output_dir):
    """Save the threshold configuration to a JSON file in the output directory."""
    config_path = Path(output_dir) / "threshold_config.json"
    config.save(config_path)
    print(f"Saved threshold configuration to {config_path}")


In [ ]:
def save_activity_class(metrics_df, summary, out_widget=sys.stdout):
    output_dir = globals()["output_dir"]

    threshold_metric = globals().get("threshold_config", {}).get("metric", "N/A")
    threshold_method = globals().get("threshold_config", {}).get("method", "N/A")

    # Save results to CSV files
    output_prefix = f"cell_activity_{threshold_metric}_{threshold_method}"

    # Detailed per-instance results
    instance_output = f"{output_dir}/{output_prefix}_per_instance.csv"
    metrics_df.to_csv(instance_output, index=False)

    # ----- Enrich summary with file paths -----
    image_paths_list = globals().get('image_paths') or []
    label_paths_list = globals().get('label_paths') or []
    activity_paths_list = globals().get('activity_paths') or []

    # Index: mCherry filename -> full mCherry path and corresponding label path
    _mcherry_by_name = {Path(p).name: Path(p) for p in image_paths_list}
    _label_by_idx = {
        Path(img_p).name: (label_paths_list[i] if i < len(label_paths_list) else None)
        for i, img_p in enumerate(image_paths_list)
    }
    # Activity lookup by filename (some images may have been skipped during save)
    _activity_by_name = {Path(p).name: Path(p) for p in activity_paths_list}

    def _build_path_row(img_name):
        mcherry_p = _mcherry_by_name.get(img_name)

        bf_path = None
        if mcherry_p is not None:
            try:
                bf_candidate = find_brightfield_from_mcherry_path(mcherry_p)
                bf_path = str(bf_candidate) if bf_candidate and bf_candidate.exists() else None
            except Exception:
                pass

        lbl_raw = _label_by_idx.get(img_name)
        lbl_path = str(lbl_raw) if lbl_raw is not None else None

        # Derive expected activity filename from the mCherry name
        act_path = None
        if mcherry_p is not None:
            act_candidate = find_activity_from_mcherry_path(
                mcherry_p,
                activity_suffix="_activity_bin",
                activity_dir=output_dir,
                must_exist=False,
            )
            # Prefer one that actually exists (may have been saved)
            if act_candidate is not None and not act_candidate.exists():
                act_candidate = _activity_by_name.get(act_candidate.name)
            act_path = str(act_candidate) if act_candidate else None

        return pd.Series({
            'mcherry_path': str(mcherry_p) if mcherry_p else None,
            'brightfield_path': bf_path,
            'label_path': lbl_path,
            'activity_path': act_path,
        })

    summary = summary.copy()
    path_cols = summary['image'].apply(_build_path_row)
    summary = pd.concat([summary, path_cols], axis=1)
    # ------------------------------------------
    if "ID" not in summary.columns:
        summary = add_meta_info(summary, file_handler=globals().get('file_handler'), id_from='image')

    # Summary per image
    summary_output = f"{output_dir}/{output_prefix}_summary.csv"
    summary.to_csv(summary_output, index=False)
    globals()['_last_summary_csv'] = summary_output

    # Overall statistics
    overall_stats = {
        'metric_used': threshold_metric,
        'threshold_method': threshold_method,
        'total_images': len(summary),
        'total_cells': len(metrics_df),
        'total_active': int(metrics_df['is_active'].sum()),
        'total_dead': int((~metrics_df['is_active']).sum()),
        'overall_activity_rate': float(metrics_df['is_active'].mean()),
        'avg_cells_per_image': float(summary['n_instances'].mean()),
        'avg_activity_rate_per_image': float(summary['activity_ratio'].mean())
    }

    import json
    stats_output = f"{output_dir}/{output_prefix}_overall_stats.json"
    with open(stats_output, 'w') as f:
        json.dump(overall_stats, f, indent=2)

    classification_results = f"""
        === CELL ACTIVITY CLASSIFICATION RESULTS ===
        Metric used: {threshold_metric}
        Threshold method: {threshold_method}
        Total images processed: {overall_stats['total_images']}
        Total cells analyzed: {overall_stats['total_cells']}
        Active cells: {overall_stats['total_active']} ({overall_stats['overall_activity_rate']:.1%})
        Dead cells: {overall_stats['total_dead']} ({1-overall_stats['overall_activity_rate']:.1%})
    """

    with out_widget:
        print(f"\nFiles saved:")
        print(f"- Per-instance results: {instance_output}")
        print(f"- Per-image summary: {summary_output}")
        print(f"- Overall statistics: {stats_output}")

        print(classification_results)


In [100]:
def validate_label_dict(label_dict):
    """Validate the label_dict format and contents."""
    if not isinstance(label_dict, dict):
        raise ValueError("label_dict must be a dictionary mapping label values to class names.")
    
    if 'active' not in label_dict or 'dead' not in label_dict:
        raise ValueError("label_dict must contain mappings for both 'active' and 'dead' classes.")
    
    for key, value in label_dict.items():
        if not isinstance(value, int):
            raise ValueError(f"Label value for key '{key}' is not an integer.")

    return True


In [102]:
import threshold_activity_classifier
print(threshold_activity_classifier.__file__)


/Users/serenasritharan/Projects/single-cell/src/threshold_activity_classifier/__init__.py


In [ ]:
def save_output_images(image_paths, output_dir, label_dict=None, out_widget=sys.stdout):
    global activity_paths

    activity_paths = []

    label_is_bin = False
    if label_dict is not None:
        try:
            validate_label_dict(label_dict)
            label_is_bin = True
        except ValueError as e:
            label_dict = None

    activity_dir = Path(output_dir)
    activity_dir.mkdir(parents=True, exist_ok=True)

    saved = 0
    skipped = 0

    metrics_df = globals().get('metrics_df')

    pairs = list(zip(image_paths, label_paths))

    with out_widget:

        for img_path, lbl_path in tqdm(pairs, desc="Saving activity labels"):
            img_p = Path(img_path)
            img_name = img_p.name
            try:
                if lbl_path is None or not lbl_path.exists():
                    print(f"Skipping {img_name}: label file not found")
                    skipped += 1
                    continue

                # Load intensity and label images (ensure 2D)
                img = tiff.imread(str(img_p))
                lbl = tiff.imread(str(lbl_path))
                img = img
                lbl = lbl.astype(np.int32, copy=False)

                # Get classification rows for this image
                classification = metrics_df[metrics_df['image'] == img_name].copy()
                if classification.empty:
                    print(f"Skipping {img_name}: no classification data available")
                    skipped += 1
                    continue

                # Create activity-labeled image (positive labels = active, negative = dead)
                activity_labels = create_activity_labeled_image(lbl, classification, label_dict=label_dict)
                activity_suffix = "_activity_bin" if label_is_bin else "_activity"
                # Determine output path for activity image to be saved
                activity_path = find_activity_from_mcherry_path(img_p, activity_suffix=activity_suffix, activity_dir=activity_dir, must_exist=False)

                # Write as int32
                tiff.imwrite(str(activity_path), activity_labels.astype(np.int32), metadata={'is_bin': label_is_bin})
                saved += 1
                activity_paths.append(activity_path)

            except Exception as e:
                print(f"Error processing {img_name}: {e}")
                skipped += 1

    print(f"Done. Saved: {saved}, Skipped/Errors: {skipped}. Files written to: {activity_dir}")

In [ ]:
# Extract sample and z-index information from image names
import re

def _update_image_viewer(csv_path=None):
    """Sync the Napari image viewer with the most recently saved activity summary CSV."""
    if csv_path is None:
        csv_path = globals().get("_last_summary_csv")
    if not csv_path:
        print("No summary CSV available. Please save activity images first.")
        return

    viewer = globals().get("image_viewer")
    if viewer is None:
        print("Warning: Image viewer not initialized. Skipping image browser update.")
        return

    viewer._csv_input.value = str(csv_path)
    viewer.load_csv(csv_path)


In [ ]:

def _on_save_images(btn):
    save_images_out.clear_output()
    save_images_summary.clear_output()

    with save_images_out:
        btn.disabled = True

        out_path = Path(output_dir_text.value).expanduser()
        out_path.mkdir(parents=True, exist_ok=True)
        globals()["output_dir"] = out_path
        print(f"Output directory: {out_path}")

        try:

            if 'image_paths' not in globals() or globals().get('image_paths') is None:
                print("No image_paths available. Please set directories and discover images first.")
                return
            if 'output_dir' not in globals():
                print("No output_dir set. Please set directories first.")
                return
            
            save_threshold_config(globals().get('threshold_config'), globals()['output_dir'])

            print("Saving activity-labeled images to:", globals()['output_dir'])
            # call the existing function (populates globals()['activity_paths'])
            save_output_images(globals()['image_paths'], globals()['output_dir'], label_dict={'active':1, 'dead':2}, out_widget=save_images_out)

            save_activity_class(globals()['metrics_df'], globals()['summary'], out_widget=save_images_summary)

            napari_sec = globals().get('napari_section')
            if napari_sec is not None:
                napari_sec.selected_index = 0

            # Auto-populate the image viewer with the saved CSV
            _update_image_viewer()

        except Exception as e:
            print("Error while saving images:", e)
        finally:
            btn.disabled = False

save_images_btn.on_click(_on_save_images)



In [111]:
if not IS_VOILA:
    display(widgets.VBox(save_widgets))

In [ ]:
from threshold_activity_classifier.ui.image_viewer_widget import ImageViewer

image_viewer = ImageViewer(file_handler=file_handler)

In [ ]:
if not IS_VOILA:
    display(image_viewer.widget)

In [ ]:
# Assemble UI

if IS_VOILA:
    # Wrap each section in its own collapsible accordion panel
    dir_section = widgets.Accordion(children=[widgets.VBox(dir_widgets)])
    dir_section.set_title(0, "1. Directory Selection & Data Filtering")
    dir_section.selected_index = 0  # open first section by default

    stats_section = widgets.Accordion(children=[widgets.VBox(compute_stats_widgets)])
    stats_section.set_title(0, "2. Cell Activity Statistics")
    stats_section.selected_index = None

    classifier_section = widgets.Accordion(children=[widgets.VBox(classifier_widgets)])
    classifier_section.set_title(0, "3. Cell Activity Thresholding")
    classifier_section.selected_index = None

    save_section = widgets.Accordion(children=[widgets.VBox(save_widgets)])
    save_section.set_title(0, "4. Export Activity Images")
    save_section.selected_index = None

    napari_section = widgets.Accordion(children=[image_viewer.widget])
    napari_section.set_title(0, "5. Visualize Images in Napari")
    napari_section.selected_index = None

    # Store in globals so callbacks can expand the next section on completion
    globals()['dir_section'] = dir_section
    globals()['stats_section'] = stats_section
    globals()['classifier_section'] = classifier_section
    globals()['save_section'] = save_section
    globals()['napari_section'] = napari_section

    ui = widgets.VBox([
        import_messages_widget,
        widgets.HTML("<h1>Cell Activity Classification Based on mCherry Images</h1>"),
        workflow_accordion,
        dir_section,
        stats_section,
        classifier_section,
        save_section,
        napari_section,
    ])

    display(ui)
